In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.losses import binary_crossentropy

# Load and preprocess the dataset
data = pd.read_csv('/content/combined_glucemia_dataset.csv')

# Preprocessing: Convert timestamps to datetime, extract hour and minute, and encode categorical variables
data['Timestamp'] = pd.to_datetime(data['Timestamp'], format='%H:%M:%S')
data['Hour'] = data['Timestamp'].dt.hour
data['Minute'] = data['Timestamp'].dt.minute
data['Condition'] = data['Condition'].apply(lambda x: 1 if x == 'Diabetes' else 0)
data['Gender'] = data['Gender'].apply(lambda x: 1 if x == 'Male' else 0)

# Drop unnecessary columns
data = data.drop(columns=['ID', 'Timestamp'])

# Separate features and target
X = data.drop(columns='Condition')
y = data['Condition']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features for deep learning models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# DEEP LEARNING MODEL: LSTM for time-series data
# Reshape for LSTM [samples, timesteps, features], assuming data can be treated as a sequence
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_test_seq = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# Build LSTM model
lstm_model = Sequential([
    LSTM(64, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, return_sequences=False),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

# Define a custom loss function to handle class imbalance
class_weights = {0: 1, 1: 2}  # Assign higher weight to minority class
def weighted_binary_crossentropy(y_true, y_pred):
    return binary_crossentropy(y_true, y_pred) * (class_weights[0] * (1 - y_true) + class_weights[1] * y_true)

lstm_model.compile(optimizer='adam', loss=weighted_binary_crossentropy, metrics=['accuracy'])

# Train the LSTM model
lstm_model.fit(X_train_seq, y_train, epochs=10, batch_size=32, validation_split=0.2)

# Evaluate the LSTM model
lstm_preds = (lstm_model.predict(X_test_seq) > 0.5).astype("int32")
print("LSTM Model Accuracy:", accuracy_score(y_test, lstm_preds))
print(classification_report(y_test, lstm_preds))

/usr/local/lib/python3.10/dist-packages/keras/src/layers/rnn/rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 116s 4ms/step - accuracy: 0.9993 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 2/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 141s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 3/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 157s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 4/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 140s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 5/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 117s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 6/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 142s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 7/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 106s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: 1.0000 - val_loss: nan
Epoch 8/10
28080/28080 ━━━━━━━━━━━━━━━━━━━━ 152s 4ms/step - accuracy: 1.0000 - loss: nan - val_accuracy: